In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from xgboost.callback import EarlyStopping

pd.set_option("display.max_rows", 200)

In [ ]:
financials = pd.read_excel(
    "../../data/processed/financials_timeseries_delta_2026-01-31.xlsx"
)
macro = pd.read_excel(
    "../../data/processed/macro_timeseries_quarterly_delta_2026-01-31.xlsx"
)

In [ ]:
financials["as_of_date"] = pd.to_datetime(financials["as_of_date"])
macro["date"] = pd.to_datetime(macro["date"])
financials = financials.sort_values("as_of_date")
macro = macro.sort_values("date")
numeric_cols = macro.select_dtypes(include="number").columns.tolist()
macro_pct = macro[numeric_cols].pct_change().add_suffix("_pct_change")
macro_delta = macro[numeric_cols].diff().add_suffix("_delta")
macro = pd.concat([macro, macro_pct, macro_delta], axis=1)
macro[macro_pct.columns] = macro[macro_pct.columns].fillna(0)
macro[macro_delta.columns] = macro[macro_delta.columns].fillna(0)

In [ ]:
financials = pd.merge_asof(
    financials,
    macro,
    left_on="as_of_date",
    right_on="date",
    direction="backward",
).rename(columns={"date": "macro_date"})

In [ ]:
ratio_defs = {
    "ebitda_margin": ("EBITDA", "Revenues"),
    "gross_profit_margin": ("Gross_Profit", "Revenues"),
    "debt_to_equity": ("Liabilities", "Stockholders_Equity"),
    "capex_to_assets": ("CapEx", "Assets"),
    "revenue_to_gdp": ("Revenues", "GDPC1"),
}
for name, (num, den) in ratio_defs.items():
    denom = financials[den].replace(0, np.nan)
    financials[name] = financials[num] / denom
financials[list(ratio_defs)] = financials[list(ratio_defs)].fillna(0)
y_col = "EBITDA"
financials = financials.replace([np.inf, -np.inf], np.nan)
financials = financials.dropna(subset=[y_col, "Revenues"]).copy()

In [ ]:
dates = np.sort(financials["as_of_date"].unique())
cutoff = dates[int(len(dates) * 0.85)]
train = financials[financials["as_of_date"] < cutoff].copy()
test = financials[financials["as_of_date"] >= cutoff].copy()

In [ ]:
train_subset = financials[financials["as_of_date"] < cutoff]
bins = pd.qcut(train_subset["Revenues"], q=5, duplicates="drop", retbins=True)[1]
financials["rev_bin"] = pd.cut(
    financials["Revenues"],
    bins=bins,
    labels=range(len(bins) - 1),
    include_lowest=True,
).astype("Int64")
fine_bins = pd.qcut(train_subset["Revenues"], q=10, duplicates="drop", retbins=True)[1]
financials["rev_bin_fine"] = pd.cut(
    financials["Revenues"],
    bins=fine_bins,
    labels=range(len(fine_bins) - 1),
    include_lowest=True,
).astype("Int64")
train_rev_counts = (
    financials.loc[financials["as_of_date"] < cutoff, "rev_bin_fine"]
    .value_counts()
    .to_dict()
)
financials["rev_bin_fine_count"] = (
    financials["rev_bin_fine"].map(train_rev_counts).fillna(0).astype("Int64")
)
train = financials[financials["as_of_date"] < cutoff].copy()
test = financials[financials["as_of_date"] >= cutoff].copy()

In [ ]:
X_num = list(macro.columns)
X_num.remove("date")
X_num += ["rev_bin"]
derived_ratio_cols = [
    "ebitda_margin",
    "gross_profit_margin",
    "debt_to_equity",
    "capex_to_assets",
    "revenue_to_gdp",
]
X_num += derived_ratio_cols
X_num += ["rev_bin_fine", "rev_bin_fine_count"]
X_cat = ["industry", "region"]
X_cols = X_num + X_cat

In [ ]:
pre = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]
            ),
            X_num,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imp", SimpleImputer(strategy="most_frequent")),
                    ("oh", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            X_cat,
        ),
    ]
)

In [ ]:
def y_to_t(y):
    return np.sign(y) * np.log1p(np.abs(y))


def t_to_y(t):
    return np.sign(t) * np.expm1(np.abs(t))


q1, q99 = train[y_col].quantile([0.01, 0.99])
y_train = train[y_col].clip(q1, q99)
y_test = test[y_col]
y_train_t = y_to_t(y_train)
y_test_t = y_to_t(y_test.clip(q1, q99))

In [ ]:
ridge = Pipeline(
    [
        ("pre", pre),
        ("reg", Ridge(alpha=2.0, random_state=0)),
    ]
)
ridge.fit(train[X_cols], y_train_t)
pred_r = t_to_y(ridge.predict(test[X_cols]))
print("Ridge R2:", r2_score(y_test, pred_r))
print("Ridge MAE:", mean_absolute_error(y_test, pred_r))

In [ ]:
xgb = Pipeline(
    [
        ("pre", pre),
        (
            "reg",
            XGBRegressor(
                callbacks=[EarlyStopping(rounds=50, save_best=True)],
                n_estimators=2000,
                learning_rate=0.05,
                max_depth=6,
                min_child_weight=5,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=2.0,
                objective="reg:squarederror",
                tree_method="hist",
                n_jobs=8,
                random_state=0,
            ),
        ),
    ]
)
eval_pre = clone(pre)
eval_pre.fit(train[X_cols])
eval_transformed = eval_pre.transform(test[X_cols])

xgb.fit(
    train[X_cols],
    y_train_t,
    reg__eval_set=[(eval_transformed, y_test_t)],
    reg__verbose=False,
)
pred_x = t_to_y(xgb.predict(test[X_cols]))
print("XGB R2:", r2_score(y_test, pred_x))
print("XGB MAE:", mean_absolute_error(y_test, pred_x))